# Thread Pools in Netrun

This notebook demonstrates how to configure and use thread pools in netrun for concurrent execution.

**Key Concepts:**
- Thread pools allow multiple workers to process nodes concurrently
- Pools are configured in the `pools` section of NetConfig
- Nodes are assigned to pools via `execution_config.pools`

**Tip:** You can visualize and edit the network configuration by running `netrun-ui` in this folder.

## Pool Types Overview

| Pool Type | Description | Best For |
|-----------|-------------|----------|
| `main` | Single worker in main event loop | Lightweight async operations |
| `thread` | Multiple worker threads | I/O-bound work, concurrent tasks |
| `multiprocess` | Separate processes | CPU-bound work (requires picklable functions) |
| `remote` | Network-connected workers | Distributed execution |

## Load the Configuration

In [1]:
import json
import time
from copy import deepcopy
from pathlib import Path

from netrun.core import Net, NetConfig

# Load the base configuration
config_path = Path("main.netrun.json")
config_data = json.loads(config_path.read_text())

print("Pool configuration from file:")
print(json.dumps(config_data["pools"], indent=2))

Pool configuration from file:
{
  "main": {
    "spec": {
      "type": "main"
    }
  },
  "thread_pool": {
    "spec": {
      "type": "thread",
      "num_workers": 4
    }
  }
}


## Helper Function to Run with Different Worker Counts

In [2]:
async def run_with_workers(base_config: dict, num_workers: int) -> float:
    """Run the network with a specific number of workers and return elapsed time."""
    config = deepcopy(base_config)
    
    # Configure thread pool with specified workers
    config["pools"] = {
        "compute_pool": {
            "spec": {"type": "thread", "num_workers": num_workers}
        },
        "main": {"spec": {"type": "main"}},
    }
    
    # Update all hash nodes to use compute_pool
    for node in config["graph"]["nodes"]:
        if node["name"].startswith("hash_"):
            node["execution_config"] = {"pools": ["compute_pool"]}
    
    net_config = NetConfig.model_validate(config)
    
    start_time = time.perf_counter()
    
    async with Net(net_config) as net:
        # Inject data for each hash node
        for i, name in enumerate(["alpha", "beta", "gamma", "delta"], 1):
            net.inject_data(f"hash_{i}", "data", [name])
            net.inject_data(f"hash_{i}", "iterations", [200_000])
        
        # Run until complete
        while True:
            await net.run_until_blocked()
            startable = net.get_startable_epochs()
            if not startable:
                break
            for epoch_id in startable:
                await net.execute_epoch(epoch_id)
        
        results = net.get_all_outputs("results")
    
    elapsed = time.perf_counter() - start_time
    return elapsed

## Run with 1 Worker (Sequential)

In [3]:
print("Running with 1 worker (sequential)...")
time_1 = await run_with_workers(config_data, 1)
print(f"Completed in {time_1:.2f}s")

Running with 1 worker (sequential)...
Completed in 0.28s


## Run with 4 Workers (Concurrent)

In [ ]:
print("Running with 4 workers (concurrent)...")
time_4 = await x(config_data, 4)
print(f"Completed in {time_4:.2f}s")

Running with 4 workers (concurrent)...
Completed in 0.27s


## Compare Results

In [5]:
print("=" * 50)
print("Performance Comparison")
print("=" * 50)
print(f"1 worker (sequential): {time_1:.2f}s")
print(f"4 workers (concurrent): {time_4:.2f}s")
print()

if time_4 < time_1:
    speedup = time_1 / time_4
    print(f"Speedup: {speedup:.2f}x faster with 4 workers")
else:
    print("Note: For CPU-bound Python code, GIL limits parallel speedup.")

Performance Comparison
1 worker (sequential): 0.28s
4 workers (concurrent): 0.27s

Speedup: 1.01x faster with 4 workers


## Understanding the GIL

Python's **Global Interpreter Lock (GIL)** prevents true parallel execution of CPU-bound Python code in threads. This means:

- **Thread pools** are best for **I/O-bound** work (network requests, file I/O)
- **Multiprocess pools** are best for **CPU-bound** work (calculations)

However, thread pools still provide benefits for:
- Concurrent I/O operations
- Operations that release the GIL (numpy, etc.)
- Overlapping computation with I/O

## Pool Configuration Reference

Here's how to configure pools in your `main.netrun.json`:

```json
{
  "pools": {
    "main": {
      "spec": {"type": "main"}
    },
    "thread_pool": {
      "spec": {
        "type": "thread",
        "num_workers": 4
      }
    }
  },
  "graph": {
    "nodes": [
      {
        "name": "my_node",
        "factory": "netrun.node_factories.from_function",
        "factory_args": {"func": "nodes.my_func"},
        "execution_config": {
          "pools": ["thread_pool"]
        }
      }
    ]
  }
}
```

## Understanding the Node Functions

Let's look at the node functions defined in `nodes.py`:

In [6]:
print(Path("nodes.py").read_text())

"""Node functions demonstrating CPU-bound work for pool comparison.

This module contains functions that perform CPU-intensive calculations.
When run in a thread pool, they are limited by Python's GIL (Global Interpreter Lock).
When run in a multiprocess pool, they can utilize multiple CPU cores in parallel.
"""

import hashlib


def compute_hash(data: str, iterations: int, print) -> dict:
    """Compute a hash iteratively (CPU-bound work).

    This simulates CPU-intensive work by repeatedly hashing a value.
    """
    print(f"Starting hash computation with {iterations} iterations")

    result = data.encode()
    for i in range(iterations):
        result = hashlib.sha256(result).digest()
        if (i + 1) % (iterations // 4) == 0:
            print(f"Progress: {(i + 1) * 100 // iterations}%")

    hex_result = result.hex()[:16]
    print(f"Completed: {hex_result}...")

    return {"input": data, "iterations": iterations, "hash": hex_result}


def is_prime(n: int) -> bool:
    """C